# Unicorn Master Evaluation Report

Comprehensive evaluation of the Unicorn contextual NBA player embedding model.

**Data source:** `eval_output/` directory produced by `scripts/precompute_eval.py`

**Sections:**
1. Training Run Summary
2. Static Embedding Quality
3. Distributional Prediction Quality
4. Attention Analysis
5. Substitution Sensitivity
6. Game Outcome Prediction & Lineup Completion
7. Archetype Clusters & Aging Curves

In [ ]:
import json
import numpy as np
import torch
import pandas as pd
from pathlib import Path

import matplotlib
matplotlib.use("agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["figure.figsize"] = (12, 5)

EVAL_DIR = Path("../eval_output_v5")
FIG_DIR = EVAL_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

OUTCOME_NAMES = [
    "made_3pt", "missed_3pt", "made_2pt_close", "made_2pt_mid",
    "missed_2pt_close", "missed_2pt_mid", "FT", "turnover_live", "turnover_dead",
]
SHORT_NAMES = ["3PM", "3PX", "2PC+", "2PM+", "2PC-", "2PM-", "FT", "TOlive", "TOdead"]

def load_json(name):
    p = EVAL_DIR / name
    if not p.exists():
        print(f"WARNING: {p} not found")
        return None
    with open(p) as f:
        return json.load(f)

summary = load_json("summary.json")
print(f"Checkpoint: {summary['checkpoint']}")
print(f"Architecture: {summary['architecture']}")
print(f"Attn temperature: {summary.get('attn_temperature', 1.0)}")
print(f"Computation time: {summary['computation_seconds']:.0f}s")

## 1. Training Run Summary

Loss curves, temporal metrics, and key training KPIs across epochs.

In [ ]:
metrics = load_json("training_metrics.json")
if metrics:
    epochs = [m["epoch"] for m in metrics]
    
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    
    # 1a: Loss components
    ax = axes[0, 0]
    ax.plot(epochs, [m["contrastive_loss"] for m in metrics], label="Contrastive", marker=".")
    ax.plot(epochs, [m["outcome_loss"] for m in metrics], label="Outcome", marker=".")
    ax.plot(epochs, [m["aux_loss"] for m in metrics], label="Aux", marker=".")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.set_title("Training Loss Components")
    ax.legend(); ax.grid(True, alpha=0.3)
    
    # 1b: Total loss (train vs val)
    ax = axes[0, 1]
    ax.plot(epochs, [m["train_loss"] for m in metrics], label="Train", marker=".")
    ax.plot(epochs, [m["val_loss"] for m in metrics], label="Val", marker=".")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.set_title("Total Loss")
    ax.legend(); ax.grid(True, alpha=0.3)
    
    # 1c: Outcome accuracy
    ax = axes[0, 2]
    ax.plot(epochs, [m["train_outcome_acc"] * 100 for m in metrics], label="Train", marker=".")
    ax.plot(epochs, [m["val_outcome_acc"] * 100 for m in metrics], label="Val", marker=".")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy (%)"); ax.set_title("Outcome Accuracy")
    ax.legend(); ax.grid(True, alpha=0.3)
    
    # 1d: Temporal metrics
    ax = axes[1, 0]
    ax.plot(epochs, [m.get("temporal_top100", 0) * 100 for m in metrics], label="Top-100", marker=".", color="green")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Hit Rate (%)"); ax.set_title("Temporal Top-100")
    ax.legend(); ax.grid(True, alpha=0.3)
    
    # 1e: Contrastive accuracy
    ax = axes[1, 1]
    ax.plot(epochs, [m.get("train_base_top1", 0) * 100 for m in metrics], label="Base Top-1", marker=".")
    ax.plot(epochs, [m.get("train_base_top5", 0) * 100 for m in metrics], label="Base Top-5", marker=".")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy (%)"); ax.set_title("Contrastive Accuracy")
    ax.legend(); ax.grid(True, alpha=0.3)
    
    # 1f: Delta norms + attention entropy
    ax = axes[1, 2]
    ax.plot(epochs, [m.get("delta_norm_mean", 0) for m in metrics], label="Delta norm", marker=".", color="orange")
    ax2 = ax.twinx()
    if "attn_entropy" in metrics[0]:
        ax2.plot(epochs, [m.get("attn_entropy", 1.609) for m in metrics], label="Attn entropy", marker=".", color="red", linestyle="--")
        ax2.axhline(y=1.6094, color="gray", linestyle=":", alpha=0.5, label="Uniform")
        ax2.set_ylabel("Entropy")
        ax2.legend(loc="upper right")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Delta Norm"); ax.set_title("Delta Norms & Attention")
    ax.legend(loc="upper left"); ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(FIG_DIR / "training_curves.png", bbox_inches="tight")
    plt.show()
    
    # Summary table
    last = metrics[-1]
    print(f"\nFinal epoch ({last['epoch']}):")
    print(f"  Train loss: {last['train_loss']:.4f}  |  Val loss: {last['val_loss']:.4f}")
    print(f"  Outcome acc: train {last['train_outcome_acc']*100:.1f}%  val {last['val_outcome_acc']*100:.1f}%")
    print(f"  Temporal top-100: {last.get('temporal_top100', 0)*100:.1f}%")
    print(f"  Delta norm: {last.get('delta_norm_mean', 0):.4f}")
    if "attn_entropy" in last:
        print(f"  Attn entropy: {last['attn_entropy']:.4f} (uniform=1.6094)")
        print(f"  Max attn weight: {last.get('max_attn_weight', 'N/A')}")
else:
    print("No training metrics available")

## 2. Static Embedding Quality

Nearest neighbors, norm distributions, same-player cross-season coherence.

In [ ]:
emb_stats = load_json("embedding_stats.json")
nn_data = load_json("nearest_neighbors.json")

if emb_stats:
    print("Embedding Statistics:")
    print(f"  Shape: {emb_stats['shape']}")
    print(f"  Norm — mean: {emb_stats['norm_mean']}, std: {emb_stats['norm_std']}, range: [{emb_stats['norm_min']}, {emb_stats['norm_max']}]")
    print(f"  Avg pairwise cosine: {emb_stats['avg_pairwise_cosine']}")
    print(f"  Same-player cross-season cosine: {emb_stats.get('same_player_cosine_mean', 'N/A')} (n={emb_stats.get('same_player_n', 0)})")

# Embedding norms distribution
emb = torch.load(EVAL_DIR / "embeddings.pt", weights_only=True)
norms = emb.norm(dim=1).numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(norms, bins=80, edgecolor="black", alpha=0.7, color="steelblue")
axes[0].set_xlabel("L2 Norm"); axes[0].set_ylabel("Count"); axes[0].set_title("Embedding Norm Distribution")
axes[0].axvline(norms.mean(), color="red", linestyle="--", label=f"mean={norms.mean():.2f}")
axes[0].legend()

# Pairwise cosine similarity histogram (sample)
import torch.nn.functional as F_plot
sample_ids = np.random.choice(len(emb), min(500, len(emb)), replace=False)
sample_emb = F_plot.normalize(emb[sample_ids], dim=1)
sim = (sample_emb @ sample_emb.T).numpy()
upper_tri = sim[np.triu_indices_from(sim, k=1)]
axes[1].hist(upper_tri, bins=80, edgecolor="black", alpha=0.7, color="coral")
axes[1].set_xlabel("Cosine Similarity"); axes[1].set_ylabel("Count"); axes[1].set_title("Pairwise Cosine Similarity (sample)")
axes[1].axvline(upper_tri.mean(), color="red", linestyle="--", label=f"mean={upper_tri.mean():.3f}")
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / "embedding_distributions.png", bbox_inches="tight")
plt.show()

In [ ]:
# Nearest neighbors table
if nn_data:
    for query, neighbors in nn_data.items():
        print(f"\n{query}:")
        for n in neighbors[:8]:
            print(f"  {n['similarity']:.4f}  {n['name']}")

## 3. Distributional Prediction Quality

Predicted vs target outcome distributions, KL divergence improvement over global baseline, per-class accuracy.

In [ ]:
dist_metrics = load_json("distributional_metrics.json")

if dist_metrics:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    x = np.arange(len(SHORT_NAMES))
    width = 0.35
    
    # 3a: Predicted vs target distribution
    ax = axes[0]
    ax.bar(x - width/2, dist_metrics["mean_target_dist"], width, label="Target", color="steelblue")
    ax.bar(x + width/2, dist_metrics["mean_pred_dist"], width, label="Predicted", color="coral")
    ax.set_xticks(x); ax.set_xticklabels(SHORT_NAMES, rotation=45, ha="right")
    ax.set_ylabel("Probability"); ax.set_title("Mean Outcome Distribution")
    ax.legend()
    
    # 3b: Pred/Target ratios
    ax = axes[1]
    ratios = dist_metrics["pred_target_ratios"]
    colors = ["green" if 0.95 <= r <= 1.05 else "orange" if 0.9 <= r <= 1.1 else "red" for r in ratios]
    ax.bar(x, ratios, color=colors, edgecolor="black", alpha=0.8)
    ax.axhline(y=1.0, color="black", linestyle="--", alpha=0.5)
    ax.axhspan(0.95, 1.05, alpha=0.1, color="green")
    ax.set_xticks(x); ax.set_xticklabels(SHORT_NAMES, rotation=45, ha="right")
    ax.set_ylabel("Pred/Target Ratio"); ax.set_title("Per-Class Calibration")
    ax.set_ylim(0.8, 1.2)
    
    # 3c: KL divergence comparison
    ax = axes[2]
    ax.bar(["Global\nBaseline", "Model"], [dist_metrics["kl_baseline"], dist_metrics["kl_model"]],
           color=["gray", "steelblue"], edgecolor="black")
    ax.set_ylabel("Cross-Entropy"); ax.set_title(f"KL Improvement: {dist_metrics['kl_improvement_pct']:.1f}%")
    
    plt.tight_layout()
    plt.savefig(FIG_DIR / "distributional_quality.png", bbox_inches="tight")
    plt.show()
    
    print(f"\nKL divergence — model: {dist_metrics['kl_model']:.5f}, baseline: {dist_metrics['kl_baseline']:.5f}")
    print(f"Improvement: {dist_metrics['kl_improvement_pct']:.1f}%")
    print(f"Hard-label accuracy: {dist_metrics['hard_label_accuracy']*100:.1f}% (argmax collapse expected)")
    print(f"Samples: {dist_metrics['n_samples']:,}")

## 4. Attention / Interaction Analysis

For v3.2 (LineupTransformer): Attention weight distributions, entropy vs uniform baseline, per-slot weight means. This is the key diagnostic for the uniform attention problem.

For v6 (RelationNetwork): No attention pooling exists. Instead, shows pairwise relation output statistics (off/def/match norms) and token diversity metrics.

In [ ]:
attn_stats = load_json("attention_stats.json")

if attn_stats:
    is_v6 = attn_stats.get("architecture") == "v6_relation"
    uniform_ent = attn_stats["uniform_entropy"]
    
    if is_v6:
        # v6: Show pairwise relation stats instead of attention analysis
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # 4a: Pairwise relation output norms
        ax = axes[0]
        categories = ["Off-Off\nPairs", "Def-Def\nPairs", "Matchup\nPairs"]
        means = [attn_stats["off_pair_norm_mean"], attn_stats["def_pair_norm_mean"],
                 attn_stats["match_pair_norm_mean"]]
        stds = [attn_stats["off_pair_norm_std"], attn_stats["def_pair_norm_std"],
                attn_stats["match_pair_norm_std"]]
        colors = ["steelblue", "coral", "mediumpurple"]
        bars = ax.bar(categories, means, yerr=stds, color=colors, edgecolor="black",
                      capsize=5, alpha=0.8)
        ax.set_ylabel("Mean Output Norm")
        ax.set_title("Pairwise Relation Output Norms (v6)")
        for bar, m in zip(bars, means):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f"{m:.3f}", ha="center", fontsize=10)
        
        # 4b: Token diversity
        ax = axes[1]
        div_mean = attn_stats["token_diversity_mean"]
        div_std = attn_stats["token_diversity_std"]
        ax.bar(["Token Diversity\n(mean pairwise cosine)"], [div_mean],
               yerr=[div_std], color="steelblue", edgecolor="black", capsize=5, alpha=0.8)
        ax.axhline(y=1.0, color="red", linestyle="--", alpha=0.5, label="Identical (1.0)")
        ax.axhline(y=0.0, color="green", linestyle="--", alpha=0.5, label="Orthogonal (0.0)")
        ax.set_ylabel("Mean Pairwise Cosine Similarity")
        ax.set_title("Encoder Token Diversity (v6)")
        ax.set_ylim(0, 1.1)
        ax.legend()
        ax.text(0, div_mean + 0.02, f"{div_mean:.4f}", ha="center", fontsize=12)
        
        plt.tight_layout()
        plt.savefig(FIG_DIR / "attention_analysis.png", bbox_inches="tight")
        plt.show()
        
        print("v6 RelationNetwork — no attention pooling (uses pairwise Relation MLPs)")
        print(f"  Off-off pair norm:  {attn_stats['off_pair_norm_mean']:.4f} +/- {attn_stats['off_pair_norm_std']:.4f}")
        print(f"  Def-def pair norm:  {attn_stats['def_pair_norm_mean']:.4f} +/- {attn_stats['def_pair_norm_std']:.4f}")
        print(f"  Matchup pair norm:  {attn_stats['match_pair_norm_mean']:.4f} +/- {attn_stats['match_pair_norm_std']:.4f}")
        print(f"  Token diversity:    {div_mean:.4f} +/- {div_std:.4f}")
        if div_mean < 0.95:
            print("  Tokens are differentiated (diversity < 0.95) -- encoder is NOT oversmoothing")
        else:
            print("  *** TOKENS ARE NEAR-IDENTICAL (diversity >= 0.95) -- encoder oversmoothing ***")
    else:
        # v3.2+: Standard attention analysis
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        # 4a: Per-slot mean weights
        ax = axes[0]
        slots = ["P1", "P2", "P3", "P4", "P5"]
        x = np.arange(5)
        width = 0.35
        ax.bar(x - width/2, attn_stats["off_mean_weights"], width, label="Offense", color="steelblue")
        ax.bar(x + width/2, attn_stats["def_mean_weights"], width, label="Defense", color="coral")
        ax.axhline(y=0.2, color="gray", linestyle="--", alpha=0.5, label="Uniform (0.2)")
        ax.set_xticks(x); ax.set_xticklabels(slots)
        ax.set_ylabel("Mean Weight"); ax.set_title("Per-Slot Attention Weights")
        ax.set_ylim(0, 0.5); ax.legend()
        
        # 4b: Entropy comparison
        ax = axes[1]
        categories = ["Off Entropy", "Def Entropy", "Uniform"]
        values = [attn_stats["off_entropy_mean"], attn_stats["def_entropy_mean"], uniform_ent]
        colors = ["steelblue", "coral", "gray"]
        ax.bar(categories, values, color=colors, edgecolor="black")
        ax.set_ylabel("Entropy (nats)"); ax.set_title("Attention Entropy vs Uniform")
        for i, v in enumerate(values):
            ax.text(i, v + 0.01, f"{v:.4f}", ha="center", fontsize=10)
        ax.set_ylim(0, uniform_ent * 1.2)
        
        # 4c: Attention weight histograms (from saved weights)
        ax = axes[2]
        try:
            attn_data = torch.load(EVAL_DIR / "attention_weights.pt", weights_only=True)
            off_w = attn_data["off_attn"].numpy().flatten()
            def_w = attn_data["def_attn"].numpy().flatten()
            ax.hist(off_w, bins=50, alpha=0.6, label="Offense", color="steelblue", density=True)
            ax.hist(def_w, bins=50, alpha=0.6, label="Defense", color="coral", density=True)
            ax.axvline(x=0.2, color="gray", linestyle="--", label="Uniform")
            ax.set_xlabel("Attention Weight"); ax.set_ylabel("Density")
            ax.set_title("Attention Weight Distribution")
            ax.legend()
        except Exception:
            ax.text(0.5, 0.5, "No attention weights file", ha="center", va="center", transform=ax.transAxes)
        
        plt.tight_layout()
        plt.savefig(FIG_DIR / "attention_analysis.png", bbox_inches="tight")
        plt.show()
        
        # Diagnostic summary
        off_gap = uniform_ent - attn_stats["off_entropy_mean"]
        def_gap = uniform_ent - attn_stats["def_entropy_mean"]
        print(f"\nEntropy gap from uniform:")
        print(f"  Offense: {off_gap:.4f} nats ({off_gap/uniform_ent*100:.2f}%)")
        print(f"  Defense: {def_gap:.4f} nats ({def_gap/uniform_ent*100:.2f}%)")
        print(f"  Max attn weight (off): {attn_stats['off_max_weight_mean']:.4f} (uniform=0.200)")
        print(f"  Max attn weight (def): {attn_stats['def_max_weight_mean']:.4f}")
        
        if off_gap < 0.05:
            print("\n  *** ATTENTION IS EFFECTIVELY UNIFORM -- pooling is mean-pooling ***")
        elif off_gap < 0.2:
            print("\n  Attention shows mild differentiation -- some player weighting")
        else:
            print("\n  Attention is meaningfully non-uniform -- good player differentiation")

## 5. Substitution Sensitivity

How much does swapping each player with a league-average replacement change the model's outcome predictions? Players are ranked by "favorability impact" — a weighted sum across outcome classes.

In [ ]:
sub_sens = load_json("substitution_sensitivity.json")

if sub_sens and sub_sens.get("players"):
    players = list(sub_sens["players"].values())
    names = [p["name"] for p in players]
    impacts = [p["impact_favorability"] for p in players]
    
    # Top 20 and Bottom 20
    n_show = min(20, len(players) // 2)
    top = players[:n_show]
    bottom = players[-n_show:]
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    # Top positive impact
    ax = axes[0]
    top_names = [p["name"].split(" (")[0] for p in top]
    top_vals = [p["impact_favorability"] for p in top]
    y_pos = np.arange(len(top_names))
    ax.barh(y_pos, top_vals, color="green", alpha=0.7, edgecolor="black")
    ax.set_yticks(y_pos); ax.set_yticklabels(top_names, fontsize=9)
    ax.set_xlabel("Favorability Impact"); ax.set_title(f"Top {n_show} Positive Impact")
    ax.invert_yaxis()
    
    # Bottom negative impact
    ax = axes[1]
    bot_names = [p["name"].split(" (")[0] for p in bottom]
    bot_vals = [p["impact_favorability"] for p in bottom]
    y_pos = np.arange(len(bot_names))
    ax.barh(y_pos, bot_vals, color="red", alpha=0.7, edgecolor="black")
    ax.set_yticks(y_pos); ax.set_yticklabels(bot_names, fontsize=9)
    ax.set_xlabel("Favorability Impact"); ax.set_title(f"Top {n_show} Negative Impact")
    ax.invert_yaxis()
    
    plt.tight_layout()
    plt.savefig(FIG_DIR / "substitution_sensitivity.png", bbox_inches="tight")
    plt.show()
    
    # Impact magnitude distribution
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(impacts, bins=60, edgecolor="black", alpha=0.7, color="steelblue")
    ax.set_xlabel("Favorability Impact"); ax.set_ylabel("Count")
    ax.set_title(f"Impact Distribution (range: [{min(impacts):.4f}, {max(impacts):.4f}])")
    ax.axvline(0, color="red", linestyle="--")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "impact_distribution.png", bbox_inches="tight")
    plt.show()
    
    print(f"Replacement player: {sub_sens['replacement_player']}")
    print(f"Players evaluated: {len(players)}")
    print(f"Impact range: [{min(impacts):.5f}, {max(impacts):.5f}]")
    print(f"Impact std: {np.std(impacts):.5f}")
else:
    print("No substitution sensitivity data available")

## 6. Game Outcome Prediction & Lineup Completion

**Game outcome**: 4-way comparison — home-always baseline, bag-of-player-IDs, static mean-pooled embeddings, contextual (transformer) embeddings.

**Lineup completion**: Hold out the 5th offensive player, score all candidates by predicted outcome favorability, report retrieval metrics.

In [ ]:
game = load_json("game_outcomes.json")
lineup = load_json("lineup_completion.json")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 6a: Game outcome prediction
ax = axes[0]
if game:
    methods = ["Home\nAlways", "Bag of\nIDs", "Static\nEmb", "Contextual\nEmb"]
    accs = [game["home_always_acc"], game["bag_of_ids_acc"],
            game["static_embedding_acc"], game["contextual_acc"]]
    colors = ["gray", "orange", "steelblue", "green"]
    bars = ax.bar(methods, [a * 100 for a in accs], color=colors, edgecolor="black")
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f"{acc*100:.1f}%", ha="center", fontsize=11, fontweight="bold")
    ax.set_ylabel("Accuracy (%)")
    ax.set_title(f"Game Win Prediction ({game['n_test_games']} test games)")
    ax.set_ylim(45, 75)
else:
    ax.text(0.5, 0.5, "No game outcome data\n(run without --skip-slow)",
            ha="center", va="center", transform=ax.transAxes)

# 6b: Lineup completion retrieval
ax = axes[1]
if lineup and lineup.get("test"):
    test_lc = lineup["test"]
    train_lc = lineup.get("train", {})
    
    k_values = [1, 5, 10, 50]
    test_recalls = [test_lc.get(f"recall_at_{k}", 0) for k in k_values]
    random_recalls = [k / test_lc.get("n_candidates", 200) for k in k_values]
    
    x = np.arange(len(k_values))
    width = 0.25
    ax.bar(x - width, [r * 100 for r in random_recalls], width, label="Random", color="gray", alpha=0.5)
    ax.bar(x, [r * 100 for r in test_recalls], width, label="Test", color="steelblue")
    if train_lc:
        train_recalls = [train_lc.get(f"recall_at_{k}", 0) for k in k_values]
        ax.bar(x + width, [r * 100 for r in train_recalls], width, label="Train", color="green", alpha=0.7)
    
    ax.set_xticks(x); ax.set_xticklabels([f"@{k}" for k in k_values])
    ax.set_xlabel("Recall@K"); ax.set_ylabel("Recall (%)")
    ax.set_title(f"Lineup Completion (MRR: test={test_lc.get('mrr', 0):.3f})")
    ax.legend()
else:
    ax.text(0.5, 0.5, "No lineup completion data\n(run without --skip-slow)",
            ha="center", va="center", transform=ax.transAxes)

plt.tight_layout()
plt.savefig(FIG_DIR / "benchmarks.png", bbox_inches="tight")
plt.show()

# Print details
if game:
    print("Game Outcome Prediction:")
    for method, acc in zip(["Home-always", "Bag-of-IDs", "Static embedding", "Contextual"], accs):
        print(f"  {method:20s}: {acc*100:.1f}%")

if lineup and lineup.get("test"):
    print(f"\nLineup Completion (test):")
    for k in [1, 5, 10, 50]:
        print(f"  Recall@{k:2d}: {test_lc[f'recall_at_{k}']*100:.1f}%  (random: {k/test_lc['n_candidates']*100:.1f}%)")
    print(f"  MRR: {test_lc['mrr']:.4f}  |  Mean rank: {test_lc['mean_rank']:.1f}  |  Median rank: {test_lc['median_rank']:.1f}")

## 7. Archetype Clusters & Aging Curves

K-means clustering on player embeddings to discover archetypes, and delta norm evolution across player ages.

In [ ]:
clusters = load_json("archetype_clusters.json")
aging = load_json("aging_curves.json")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 7a: Cluster sizes
ax = axes[0]
if clusters:
    sizes = clusters["cluster_sizes"]
    ax.bar(range(len(sizes)), sizes, color="steelblue", edgecolor="black")
    ax.set_xlabel("Cluster ID"); ax.set_ylabel("Players")
    ax.set_title(f"Archetype Clusters (K={clusters['k']}, silhouette={clusters['silhouette_score']:.3f})")
    for i, s in enumerate(sizes):
        ax.text(i, s + 2, str(s), ha="center", fontsize=9)
else:
    ax.text(0.5, 0.5, "No cluster data", ha="center", va="center", transform=ax.transAxes)

# 7b: Aging curves
ax = axes[1]
if aging:
    ages = sorted([int(k) for k in aging.keys()])
    means = [aging[str(a)]["mean_delta_norm"] for a in ages]
    stds = [aging[str(a)]["std_delta_norm"] for a in ages]
    counts = [aging[str(a)]["n_players"] for a in ages]
    
    ax.plot(ages, means, marker="o", color="steelblue", linewidth=2)
    ax.fill_between(ages, 
                     [m - s for m, s in zip(means, stds)],
                     [m + s for m, s in zip(means, stds)],
                     alpha=0.2, color="steelblue")
    ax.set_xlabel("Approximate Age"); ax.set_ylabel("Mean Delta Norm")
    ax.set_title("Aging Curve (Delta Deviation from Base)")
    
    # Show counts as secondary axis
    ax2 = ax.twinx()
    ax2.bar(ages, counts, alpha=0.15, color="gray", width=0.8)
    ax2.set_ylabel("N players (bars)", color="gray")
    ax2.tick_params(axis="y", labelcolor="gray")
else:
    ax.text(0.5, 0.5, "No aging data", ha="center", va="center", transform=ax.transAxes)

plt.tight_layout()
plt.savefig(FIG_DIR / "clusters_aging.png", bbox_inches="tight")
plt.show()

# Cluster members
if clusters:
    print("Archetype cluster samples:")
    for cid, members in clusters["cluster_members"].items():
        sample = ", ".join(members[:8])
        print(f"  Cluster {cid} ({clusters['cluster_sizes'][int(cid)]} players): {sample}...")

## Summary

Key metrics at a glance for this checkpoint, and cross-run comparison table.

In [ ]:
if summary:
    print("=" * 60)
    print("EVALUATION SUMMARY")
    print("=" * 60)
    print(f"  Checkpoint:         {summary['checkpoint']}")
    print(f"  Architecture:       {summary['architecture']}")
    print(f"  Attn temperature:   {summary.get('attn_temperature', 1.0)}")
    print(f"  Epochs trained:     {summary.get('num_epochs_trained', '?')}")
    print()
    print(f"  Embedding shape:    {summary['embedding_shape']}")
    print(f"  Pairwise cosine:    {summary['avg_pairwise_cosine']}")
    print(f"  Same-player cosine: {summary.get('same_player_cosine', 'N/A')}")
    print()
    print(f"  KL improvement:     {summary.get('kl_improvement_pct', 'N/A')}%")
    print(f"  Off attn entropy:   {summary.get('off_attn_entropy', 'N/A')} (uniform=1.6094)")
    print(f"  Def attn entropy:   {summary.get('def_attn_entropy', 'N/A')}")
    print(f"  Max attn weight:    {summary.get('max_attn_weight', 'N/A')}")
    print()
    if summary.get("final_val_loss"):
        print(f"  Final val loss:     {summary['final_val_loss']:.4f}")
    if summary.get("final_val_outcome_acc"):
        print(f"  Final val accuracy: {summary['final_val_outcome_acc']*100:.1f}%")
    if summary.get("final_temporal_top100"):
        print(f"  Final temporal@100: {summary['final_temporal_top100']*100:.1f}%")
    print()
    print(f"  Silhouette score:   {summary.get('silhouette_score', 'N/A')}")
    print(f"  Substitution players: {summary.get('n_substitution_players', 'N/A')}")
    
    go = summary.get("game_outcome", {})
    if go:
        print()
        print(f"  Game outcome (contextual): {go.get('contextual_acc', 'N/A')}")
    
    lc = summary.get("lineup_completion", {})
    if lc and lc.get("test"):
        print(f"  Lineup MRR (test):  {lc['test'].get('mrr', 'N/A')}")
    
    print("=" * 60)

### Cross-Run Comparison

Load multiple `summary.json` files to compare across ablation runs or model versions.

In [ ]:
# Cross-run comparison — add paths to summary.json files from different runs
comparison_dirs = [
    # ("v3.2 control", Path("../eval_output_control")),
    # ("v4 temp", Path("../eval_output_temp")),
    # ("v4 entropy", Path("../eval_output_entropy")),
]

if comparison_dirs:
    rows = []
    for label, d in comparison_dirs:
        s = json.loads((d / "summary.json").read_text()) if (d / "summary.json").exists() else {}
        rows.append({
            "Run": label,
            "Attn Temp": s.get("attn_temperature", 1.0),
            "Off Entropy": s.get("off_attn_entropy", "?"),
            "Max Attn Wt": s.get("max_attn_weight", "?"),
            "KL Improv %": s.get("kl_improvement_pct", "?"),
            "Val Loss": s.get("final_val_loss", "?"),
            "Temporal@100": s.get("final_temporal_top100", "?"),
            "Same-Player Cos": s.get("same_player_cosine", "?"),
            "Game Outcome": s.get("game_outcome", {}).get("contextual_acc", "?"),
        })
    
    comparison_df = pd.DataFrame(rows)
    display(comparison_df)
else:
    print("No comparison runs configured. Uncomment paths above after running ablations.")